# GLMM


**Block3**


In [1]:
import os
import pandas as pd
import numpy as np
import re
import ast

Agents = ['A','B','C']

def get_sorted_csv_files(directory_path):
    csv_files = [file for file in os.listdir(directory_path) if file.endswith('.csv')]
    csv_files.sort(key=lambda x: int(re.search(r'\d+', x).group()))
    
    return csv_files


def normalize(df, column_name):

    def min_max_normalize(x):
        min_val = x.min()
        max_val = x.max()
        if min_val == max_val:
            return np.ones_like(x) 
        return (x - min_val) / (max_val - min_val)

    df[column_name] = df.groupby('subject_id')[column_name].transform(min_max_normalize)
    return df

def calculate_mean(row, df):
    subset_df = df.loc[:row.name]
    mask = subset_df['choice'] == row['agent_choices']
    # print(mask)
    matching_outcomes = subset_df.loc[mask, 'outcome']
    # print(matching_outcomes)
    if matching_outcomes.empty:
        # print('zero')
        return 0 

    # print(matching_outcomes.mean())
    return matching_outcomes.mean()

def calculate_Q(row, df, index_num):
    subset_df = df.loc[:row.name]
    mask = subset_df['choice'] == index_num
    matching_outcomes = subset_df.loc[mask, 'outcome']
    
    if matching_outcomes.empty:
        return 0

    return matching_outcomes.mean()

def process_and_combine_files(directory_path,group_type,start_point):
    sorted_csv_files = get_sorted_csv_files(directory_path)
    data_frames = []

    for subject_id, file in enumerate(sorted_csv_files):
        file_path = os.path.join(directory_path, file)
        df = pd.read_csv(file_path)[45:]
        df['outcome'] = (df['outcome'] == 'win') * 1
        df['subject_id'] = subject_id + start_point
        df['group'] = group_type
        df['agent types'] = df['agent types'].apply(lambda x: ast.literal_eval(x))
        df['best_agent'] = df['agent types'].apply(lambda x: Agents[x.index(0)])
        df['second_best_agent'] = df['agent types'].apply(lambda x: Agents[x.index(1)])
        df['random_agent'] = df['agent types'].apply(lambda x: Agents[x.index(2)])

        df.loc[df['chosen_agent'] == 'ٔNo agent', 'agent_choices'] = -1
        df['agent_choices'] = df['agent_choices'].replace('-', pd.NA)
        df['agent_choices'] = pd.to_numeric(df['agent_choices']) 
        df['agent_choices'] = df['agent_choices'].fillna(method='ffill')

        
        df['chosen_agent'] = df['chosen_agent'].replace('-', pd.NA)
        df['chosen_agent'] = df['chosen_agent'].fillna(method='ffill')
        df['ag_choice_rt'] = df['ag_choice_rt'].replace('-', pd.NA)
        df['ag_choice_rt'] = pd.to_numeric(df['ag_choice_rt']) 
        df['ag_choice_rt'] = df['ag_choice_rt'].fillna(method='ffill')
#         print("subkect id is:",subject_id + start_point)

        df['choose_best_agent'] = np.where((df['best_agent'] == df['chosen_agent']), 1, 0)
        df['choose_random_agent'] = np.where((df['random_agent'] == df['chosen_agent']), 1, 0)
        df['choose_second_best_agent'] = np.where((df['second_best_agent'] == df['chosen_agent']), 1, 0)
        df['choose_two_best_agent'] = ( df['choose_best_agent'] | df['choose_second_best_agent'])*1

        df['card probs'] = df['card probs'].apply(lambda x: ast.literal_eval(x))
        df['arg_07'] = df['card probs'].apply(lambda x: x.index(0.7))
        df['arg_06'] = df['card probs'].apply(lambda x: x.index(0.6))
        df['arg_05'] = df['card probs'].apply(lambda x: x.index(0.5))
        df['arg_04'] = df['card probs'].apply(lambda x: x.index(0.4))

        df['choose_two_best_card'] = np.where((df['choice'] == df['arg_07']) | (df['choice'] == df['arg_06']) , 1, 0) 
        df['choose_best_card'] = np.where((df['choice'] == df['arg_07']), 1, 0) 
        df['choose_second_best_card'] = np.where((df['choice'] == df['arg_06']), 1, 0) 

        df['agent_choose_two_best_card'] = np.where((df['agent_choices'] == df['arg_07']) | (df['agent_choices'] == df['arg_06']) , 1, -1)
        df['two_best_agent_choose_two_best_card'] = df['choose_two_best_agent']*df['agent_choose_two_best_card']
        df['imitate'] = ((df['choice'] == df['agent_choices'])*1)
        df['imitate_from_best_agent'] = (df['imitate'] & df['choose_best_agent'] )*1
        df['imitate_from_second_best_agent'] = (df['imitate'] & df['choose_second_best_agent'] )*1
        df['imitate_from_random_agent'] = (df['imitate'] & df['choose_random_agent'] )*1
        df['imitiate_from_two_best_agent'] = (df['imitate_from_best_agent']| df['imitate_from_second_best_agent'] ).values.astype(int)
        df['last_choice_same'] = (df['choice'].shift(1) == df['agent_choices']) * 1
        df['conf_increase'] = (df['imitate'] & (df['confidence_level'].shift(1) <= df['confidence_level']))*1
        df['assimilation_rate'] = df.apply(lambda row: ((row['agent_choices'] == df.loc[:row['trial'],'choice'])*1).sum()/(row['trial']-44), axis=1)
        df['Q_agent_choice'] = df.apply(lambda row: calculate_mean(row, df), axis=1)
        df = normalize(df, 'confidence_level')
        df = normalize(df, 'ag_choice_rt')
        df['meta_cognition'] = np.where((df['confidence_level'] >= 0.5) , 1, -1)
        df['meta_cognition'] = df['meta_cognition'] * df['choose_two_best_card']
        df['ag_choice_rt_and_group'] = df['ag_choice_rt']*df['group']
        df['Q_cards'] = df.apply(lambda row: [calculate_Q(row, df, i) for i in range(4)], axis=1)
        df['Q_relative_chosen_card']= df.apply(lambda row: calculate_Q(row, df, row['choice'])-((np.sum(row['Q_cards'])- calculate_Q(row, df, row['choice']))/3) , axis=1)
        df['trial'] = pd.to_numeric(df['trial'])
       

        data_frames.append(df)
    combined_df = pd.concat(data_frames, ignore_index=True)
    return combined_df

def compute_correlations(df):
    correlations = df.corr(numeric_only=True)
    return correlations





**Informed VS Not-Informed**

In [2]:
directory_path_trained = 'D:/University/University of Tehran/MSc/Thesis/task/data/trained/4 arm'
combined_df_trained = process_and_combine_files(directory_path_trained, 0,0)
directory_path_not_trained = 'D:/University/University of Tehran/MSc/Thesis/task/data/not trained/4 arm'
combined_df_not_trained = process_and_combine_files(directory_path_not_trained, 1,45)
final_combined_df = pd.concat([combined_df_trained, combined_df_not_trained], ignore_index=True)


In [3]:
correlation_matrix = compute_correlations(final_combined_df)

In [4]:
correlation_matrix['assimilation_rate']

trial                                 -0.234119
choice                                -0.009434
outcome                                0.053396
choice_rt                             -0.046848
confidence_level                       0.077579
confidence_rt                          0.044383
agent_choices                          0.352434
ag_choice_rt                           0.124773
trial_type                                  NaN
next_trial_rt                          0.044305
subject_id                            -0.122823
group                                 -0.169147
choose_best_agent                      0.441635
choose_random_agent                   -0.090713
choose_second_best_agent               0.044720
choose_two_best_agent                  0.478253
arg_07                                 0.013097
arg_06                                -0.075016
arg_05                                 0.069357
arg_04                                -0.007644
choose_two_best_card                   0

In [5]:
final_combined_df.isna().sum()

trial                                  0
choice                                 0
outcome                                0
choice_rt                              0
confidence_level                       0
confidence_rt                          0
chosen_agent                           0
agent_choices                          0
ag_choice_rt                           0
trial_type                             0
next_trial_rt                          0
watch_agent_rt                         0
card probs                             0
agent types                            0
subject_id                             0
group                                  0
best_agent                             0
second_best_agent                      0
random_agent                           0
choose_best_agent                      0
choose_random_agent                    0
choose_second_best_agent               0
choose_two_best_agent                  0
arg_07                                 0
arg_06          

In [6]:

final_combined_df[0:50]
#print(final_combined_df['agent_choices'].isna().sum())

,trial,choice,outcome,choice_rt,confidence_level,confidence_rt,chosen_agent,agent_choices,ag_choice_rt,trial_type,...,imitate_from_random_agent,imitiate_from_two_best_agent,last_choice_same,conf_increase,assimilation_rate,Q_agent_choice,meta_cognition,ag_choice_rt_and_group,Q_cards,Q_relative_chosen_card
0,45,2,1,0.568393,1.0,0.907884,A,2.0,0.656763,3,...,1,0,0,0,1.000000,1.000000,0,0.0,"[0, 0, 1.0, 0]",1.000000
1,46,0,1,5.337687,0.5,1.053551,A,2.0,0.656763,3,...,0,0,1,0,0.500000,1.000000,1,0.0,"[1.0, 0, 1.0, 0]",0.666667
2,47,0,1,2.902809,0.5,1.615126,A,2.0,0.656763,3,...,0,0,0,0,0.333333,1.000000,1,0.0,"[1.0, 0, 1.0, 0]",0.666667
3,48,1,1,0.346040,1.0,0.972394,A,1.0,0.254121,3,...,1,0,0,1,0.250000,1.000000,0,0.0,"[1.0, 1.0, 1.0, 0]",0.333333
4,49,2,0,3.034781,0.5,1.204806,A,1.0,0.254121,3,...,0,0,1,0,0.200000,1.000000,0,0.0,"[1.0, 1.0, 0.5, 0]",-0.166667
5,50,0,0,1.842427,0.5,0.713826,A,1.0,0.254121,3,...,0,0,0,0,0.166667,1.000000,1,0.0,"[0.6666666666666666, 1.0, 0.5, 0]",0.166667
6,51,2,0,0.326801,1.0,1.880838,A,2.0,0.125669,3,...,1,0,0,1,0.428571,0.333333,0,0.0,"[0.6666666666666666, 1.0, 0.3333333333333333, 0]",-0.222222
7,52,3,1,1.329557,0.0,1.108959,A,2.0,0.125669,3,...,0,0,1,0,0.375000,0.333333,-1,0.0,"[0.6666666666666666, 1.0, 0.3333333333333333, ...",0.333333
8,53,0,1,1.392395,0.5,0.800803,A,2.0,0.125669,3,...,0,0,0,0,0.333333,0.333333,1,0.0,"[0.75, 1.0, 0.3333333333333333, 1.0]",-0.027778
9,54,3,1,0.311509,0.5,1.951136,A,3.0,0.225081,3,...,1,0,0,1,0.200000,1.000000,1,0.0,"[0.75, 1.0, 0.3333333333333333, 1.0]",0.305556


In [12]:
import os
import pandas as pd
os.environ["C:/PROGRA~1/R/R-46~1.1"] = r"C:\Program Files\R\R-YOUR-VERSION"
os.environ["RPY2_CFFI_MODE"] = "ABI"

In [13]:
%pip install "numpy<2" "pandas<2.2" "rpy2==3.5.12" "pymer4==0.8.2"

  Using cached rpy2-3.5.12.tar.gz (217 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for rpy2: filename=rpy2-3.5.12-py3-none-any.whl size=219817 sha256=ec711ad2c303803aeebb2ea4897cef02b3d132cfe5b2b4cf82dbfdd2e0346906
  Stored in directory: c:\users\asusiran\appdata\local\pip\cache\wheels\1c\27\52\e5b93aa490658b38594e7890de696a304e0af5f7fe65688aa8
Successfully built rpy2
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
from pathlib import Path

r_home = Path(r"C:/PROGRA~1/R/R-46~1.1")
assert r_home.is_dir(), f"Incorrect R path: {r_home}"

os.environ["R_HOME"] = str(r_home)
os.environ["RPY2_CFFI_MODE"] = "ABI"
os.environ["PATH"] = (
    str(r_home / "bin" / "x64")
    + os.pathsep
    + os.environ["PATH"]
)

from pymer4.models import Lmer
import pandas as pd

print("Lmer imported successfully")

Lmer imported successfully


# Card Selection

In [9]:
formula = 'choose_best_card ~ Q_relative_chosen_card + choose_second_best_agent + choose_best_agent + choose_best_agent:group +  choose_second_best_agent:group + Q_relative_chosen_card:group + group + (1|subject_id)'
model = Lmer(formula, data=final_combined_df, family='binomial')
result = model.fit()
print(result)

c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '


Linear mixed model fit by maximum likelihood  ['lmerMod']
Formula: choose_best_card~Q_relative_chosen_card+choose_second_best_agent+choose_best_agent+choose_best_agent:group+choose_second_best_agent:group+Q_relative_chosen_card:group+group+(1|subject_id)

Family: binomial	 Inference: parametric

Number of observations: 5400	 Groups: {'subject_id': 90.0}

Log-likelihood: -2694.286 	 AIC: 5406.573

Random effects:

                   Name    Var    Std
subject_id  (Intercept)  0.794  0.891

No random effect correlations specified

Fixed effects:

                                Estimate  2.5_ci  97.5_ci     SE      OR  \
(Intercept)                       -1.433  -1.768   -1.098  0.171   0.239   
Q_relative_chosen_card             3.600   3.205    3.995  0.202  36.614   
choose_second_best_agent          -0.493  -0.796   -0.190  0.155   0.611   
choose_best_agent                  1.342   1.064    1.619  0.142   3.826   
group                              0.227  -0.240    0.694  0.238   1.

In [10]:
formula = 'choose_second_best_card ~ Q_relative_chosen_card + choose_second_best_agent + choose_best_agent + choose_best_agent:group +  choose_second_best_agent:group + Q_relative_chosen_card:group + group + (1|subject_id)'
model = Lmer(formula, data=final_combined_df, family='binomial')
result = model.fit()
print(result)

c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '


Linear mixed model fit by maximum likelihood  ['lmerMod']
Formula: choose_second_best_card~Q_relative_chosen_card+choose_second_best_agent+choose_best_agent+choose_best_agent:group+choose_second_best_agent:group+Q_relative_chosen_card:group+group+(1|subject_id)

Family: binomial	 Inference: parametric

Number of observations: 5400	 Groups: {'subject_id': 90.0}

Log-likelihood: -2753.621 	 AIC: 5525.241

Random effects:

                   Name    Var    Std
subject_id  (Intercept)  0.784  0.886

No random effect correlations specified

Fixed effects:

                                Estimate  2.5_ci  97.5_ci     SE     OR  \
(Intercept)                       -1.115  -1.439   -0.792  0.165  0.328   
Q_relative_chosen_card            -0.345  -0.663   -0.027  0.162  0.708   
choose_second_best_agent           0.729   0.458    0.999  0.138  2.072   
choose_best_agent                 -0.837  -1.145   -0.529  0.157  0.433   
group                             -0.171  -0.626    0.283  0.232  0

**Imitation**

In [11]:
formula = 'imitate ~ Q_relative_chosen_card + choose_second_best_agent + choose_best_agent + choose_best_agent:group + choose_second_best_agent:group + group + (1|subject_id)'
model = Lmer(formula, data=final_combined_df, family='binomial')
result = model.fit()
print(result)

c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '


Linear mixed model fit by maximum likelihood  ['lmerMod']
Formula: imitate~Q_relative_chosen_card+choose_second_best_agent+choose_best_agent+choose_best_agent:group+choose_second_best_agent:group+group+(1|subject_id)

Family: binomial	 Inference: parametric

Number of observations: 5400	 Groups: {'subject_id': 90.0}

Log-likelihood: -2876.245 	 AIC: 5768.491

Random effects:

                   Name    Var    Std
subject_id  (Intercept)  1.673  1.294

No random effect correlations specified

Fixed effects:

                                Estimate  2.5_ci  97.5_ci     SE     OR  \
(Intercept)                       -0.277  -0.698    0.144  0.215  0.758   
Q_relative_chosen_card             0.674   0.462    0.885  0.108  1.961   
choose_second_best_agent           0.912   0.638    1.185  0.140  2.489   
choose_best_agent                  1.713   1.445    1.981  0.137  5.545   
group                             -1.002  -1.598   -0.406  0.304  0.367   
choose_best_agent:group           -0.

In [12]:
formula = 'imitate ~ assimilation_rate + assimilation_rate:group + Q_relative_chosen_card + choose_best_agent + choose_second_best_agent + group + (1|subject_id)'
model = Lmer(formula, data=final_combined_df, family='binomial')
result = model.fit()
print(result)

c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '


Linear mixed model fit by maximum likelihood  ['lmerMod']
Formula: imitate~assimilation_rate+assimilation_rate:group+Q_relative_chosen_card+choose_best_agent+choose_second_best_agent+group+(1|subject_id)

Family: binomial	 Inference: parametric

Number of observations: 5400	 Groups: {'subject_id': 90.0}

Log-likelihood: -2481.696 	 AIC: 4979.393

Random effects:

                   Name    Var    Std
subject_id  (Intercept)  1.377  1.173

No random effect correlations specified

Fixed effects:

                          Estimate  2.5_ci  97.5_ci     SE       OR  \
(Intercept)                 -1.246  -1.649   -0.843  0.206    0.288   
assimilation_rate            4.796   4.255    5.337  0.276  121.041   
Q_relative_chosen_card       0.117  -0.118    0.352  0.120    1.124   
choose_best_agent            0.491   0.268    0.713  0.114    1.633   
choose_second_best_agent     0.411   0.196    0.626  0.109    1.508   
group                       -0.931  -1.493   -0.370  0.287    0.394   
ass

**Confidence**

In [13]:
formula = 'confidence_level ~ choose_best_agent + Q_relative_chosen_card + Q_relative_chosen_card:group + choose_best_agent:group + group + (1|subject_id)'
model = Lmer(formula, data=final_combined_df)
result = model.fit()
print(result)

c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '


Linear mixed model fit by REML [’lmerMod’]
Formula: confidence_level~choose_best_agent+Q_relative_chosen_card+Q_relative_chosen_card:group+choose_best_agent:group+group+(1|subject_id)

Family: gaussian	 Inference: parametric

Number of observations: 5400	 Groups: {'subject_id': 90.0}

Log-likelihood: -954.309 	 AIC: 1924.618

Random effects:

                   Name    Var    Std
subject_id  (Intercept)  0.031  0.175
Residual                 0.079  0.280

No random effect correlations specified

Fixed effects:

                              Estimate  2.5_ci  97.5_ci     SE        DF  \
(Intercept)                      0.546   0.492    0.600  0.027    98.602   
choose_best_agent                0.072   0.045    0.100  0.014  5372.249   
Q_relative_chosen_card           0.130   0.095    0.166  0.018  5372.385   
group                           -0.026  -0.102    0.050  0.039    97.065   
Q_relative_chosen_card:group     0.028  -0.021    0.077  0.025  5372.967   
choose_best_agent:group    

In [14]:
formula = 'confidence_level ~ choose_best_agent + Q_relative_chosen_card + assimilation_rate + choose_best_agent:group + Q_relative_chosen_card:group + assimilation_rate:group + group + (1|subject_id)'
model = Lmer(formula, data=final_combined_df)
result = model.fit()
print(result)

c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '


Linear mixed model fit by REML [’lmerMod’]
Formula: confidence_level~choose_best_agent+Q_relative_chosen_card+assimilation_rate+choose_best_agent:group+Q_relative_chosen_card:group+assimilation_rate:group+group+(1|subject_id)

Family: gaussian	 Inference: parametric

Number of observations: 5400	 Groups: {'subject_id': 90.0}

Log-likelihood: -951.073 	 AIC: 1922.145

Random effects:

                   Name    Var    Std
subject_id  (Intercept)  0.030  0.173
Residual                 0.078  0.280

No random effect correlations specified

Fixed effects:

                              Estimate  2.5_ci  97.5_ci     SE        DF  \
(Intercept)                      0.527   0.473    0.582  0.028   108.135   
choose_best_agent                0.053   0.023    0.083  0.015  5381.372   
Q_relative_chosen_card           0.111   0.073    0.148  0.019  5374.297   
assimilation_rate                0.077   0.029    0.125  0.024  5376.279   
group                            0.008  -0.068    0.085  0.03

# Robustness Analyses

## Participant-Level Cluster Bootstrap

In [18]:
formula_card = (
    'choose_best_card ~ '
    'Q_relative_chosen_card + '
    'choose_second_best_agent + '
    'choose_best_agent + '
    'choose_best_agent:group + '
    'choose_second_best_agent:group + '
    'Q_relative_chosen_card:group + '
    'group + '
    '(1|subject_id)'
)

formula_imitation = (
    'imitate ~ '
    'assimilation_rate + '
    'assimilation_rate:group + '
    'Q_relative_chosen_card + '
    'choose_best_agent + '
    'choose_second_best_agent + '
    'group + '
    '(1|subject_id)'
)

formula_confidence = (
    'confidence_level ~ '
    'choose_best_agent + '
    'Q_relative_chosen_card + '
    'assimilation_rate + '
    'choose_best_agent:group + '
    'Q_relative_chosen_card:group + '
    'assimilation_rate:group + '
    'group + '
    '(1|subject_id)'
)

In [19]:
rng = np.random.default_rng(12345)

def cluster_bootstrap_df(
    data,
    subject_col='subject_id',
    group_col='group'
):
    """
    Participant-level cluster bootstrap.

    Participants are sampled with replacement separately
    within SoLI and SoLU. All trials belonging to a sampled
    participant are retained together.
    """

    pieces = []
    new_subject_id = 0

    for group_value in sorted(data[group_col].unique()):

        group_data = data[
            data[group_col] == group_value
        ]

        subjects = group_data[
            subject_col
        ].unique()

        sampled_subjects = rng.choice(
            subjects,
            size=len(subjects),
            replace=True
        )

        for subject in sampled_subjects:

            subject_data = group_data[
                group_data[subject_col] == subject
            ].copy()

            # A participant may be selected multiple times.
            # Each bootstrap copy must therefore receive
            # a unique cluster ID.
            subject_data[subject_col] = new_subject_id

            pieces.append(subject_data)

            new_subject_id += 1

    return pd.concat(
        pieces,
        ignore_index=True
    )

### Bootstrap: Optimal Card Selection Model

In [ ]:
n_boot = 1000

bootstrap_card = []

for b in range(n_boot):

    boot_df = cluster_bootstrap_df(
        final_combined_df
    )

    try:

        boot_model = Lmer(
            formula_card,
            data=boot_df,
            family='binomial'
        )

        boot_result = boot_model.fit(
            summarize=False
        )

        beta = boot_result.loc[
            'choose_best_agent:group',
            'Estimate'
        ]

        bootstrap_card.append(beta)

    except Exception:
        pass

bootstrap_card = np.asarray(
    bootstrap_card
)

print(
    "Successful fits:",
    len(bootstrap_card),
    "/",
    n_boot
)

print(
    "Median beta:",
    np.median(bootstrap_card)
)

print(
    "95% bootstrap CI:",
    np.percentile(
        bootstrap_card,
        [2.5, 97.5]
    )
)

print(
    "Proportion beta < 0:",
    np.mean(bootstrap_card < 0)
)

c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to conve

Model failed to converge with max|grad| = 0.00675661 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting. 



c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to conve

Model failed to converge with max|grad| = 0.00427344 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting. 



c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to conve

Model failed to converge with max|grad| = 0.00345467 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting. 



... (9 outputs left out)

Model failed to converge with max|grad| = 0.002267 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting. 



c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to conve

Model failed to converge with max|grad| = 0.00586312 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting. 



c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "card probs". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "agent types". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "Q_cards". Fall back to string conversion. The error is: <class 'list'>
  warnings.warn('Error while trying to convert '
c:\Users\AsusIran\AppData\Local\Programs\Python\Python311\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to conve

### Bootstrap: Imitation Model

In [ ]:
bootstrap_imitation = []

for b in range(n_boot):

    boot_df = cluster_bootstrap_df(
        final_combined_df
    )

    try:

        boot_model = Lmer(
            formula_imitation,
            data=boot_df,
            family='binomial'
        )

        boot_result = boot_model.fit(
            summarize=False
        )

        beta = boot_result.loc[
            'group',
            'Estimate'
        ]

        bootstrap_imitation.append(beta)

    except Exception:
        pass

bootstrap_imitation = np.asarray(
    bootstrap_imitation
)

print(
    "Successful fits:",
    len(bootstrap_imitation),
    "/",
    n_boot
)

print(
    "Median beta:",
    np.median(bootstrap_imitation)
)

print(
    "95% bootstrap CI:",
    np.percentile(
        bootstrap_imitation,
        [2.5, 97.5]
    )
)

print(
    "Proportion beta < 0:",
    np.mean(bootstrap_imitation < 0)
)